In [ ]:
!pip install -q matplotlib seaborn pandas wordcloud

In [ ]:
import json
import re
from collections import Counter, defaultdict
import os
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pandas as pd
from wordcloud import WordCloud
from google.colab import files
from google.colab import drive
drive.mount('/content/drive/')
train_set_path='/content/drive/MyDrive/data/outputs/drivelm_nusc.json'
with open(train_set_path,'r') as f:
  entries = json.load(f)
print(f"DriveLM enteries:{len(train_set)}")

In [ ]:
######### get the split of QA pairs under each category ###############
QA_CATEGORIES = ["perception", "prediction", "planning"]

rows = []
for entry in entries:
    qa_paits = entry.get("qa_paits", {})
    if isinstance(qa_paits, dict):
        for cat in QA_CATEGORIES:
            for qa in qa_paits.get(cat, []):
                q = (qa.get("Q") or "").strip()
                a = (qa.get("A") or "").strip()
                if q and a:
                    rows.append({
                        "question":  q,
                        "answer":    a,
                        "category":  cat,
                    })

print(f"samples : {len(rows)}")
print(f"Category counts  : "
      f"{sum(1 for r in rows if r['category']=='perception')} perception | "
      f"{sum(1 for r in rows if r['category']=='prediction')} prediction | "
      f"{sum(1 for r in rows if r['category']=='planning')} planning")

In [ ]:

df = pd.DataFrame(rows)
print(f"Loaded {len(df):,} QA pairs")
assert len(df) > 0, "No QA pairs found — check your JSON schema and update extract_records()."



CATEGORY_PATTERNS = {
    "Object Detection":   r"\b(car|vehicle|truck|bus|pedestrian|cyclist|object|detect)\b",
    "Traffic":      r"\b(sign|signal|light|traffic light|red|green|yellow|stop)\b",
    "Ego Behaviour":      r"\b(turn|lane|brake|accelerat|slow|speed|maneuv|park)\b",
    "Spatial / Distance": r"\b(far|close|near|distance|left|right|front|behind|ahead|behind)\b",
    "Scene Understanding":r"\b(weather|rain|night|day|scene|environment|road|visibility)\b",
    "Road Understanding":r"\b(lane|road|intersection)\b",
    "Prediction":         r"\b(will|would|next|future|predict|intent|plan)\b",
    "Attribute_recognition": r"\b(color|shape|size|type of)\b",
    "Counting":           r"\b(how many|count|number of|number)\b",
}

## category of question
def get_category(q: str) -> str:
    q_lower = q.lower()
    for cat, pattern in CATEGORY_PATTERNS.items():
        if re.search(pattern, q_lower):
            return cat
    return "Other"

df["category"] = df["question"].apply(get_category)

# answer type
def get_answer_type(a: str) -> str:
    a = a.strip().lower()
    if a in {"yes", "no"}:
        return "Yes/No"
    if re.fullmatch(r"-?\d+(\.\d+)?", a):
        return "Numeric"
    if len(a.split()) <= 5:
        return "Short phrase"
    return "Long description"

df["answer_type"] = df["answer"].apply(get_answer_type)


df["q_len"] = df["question"].apply(lambda x: len(x.split()))
df["a_len"] = df["answer"].apply(lambda x: len(x.split()))


sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
FIG_COLS = "#4C72B0"

def save_and_show(fig, name):
    fig.savefig(f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {name}.png")



#Semantic category
cat_counts = df["category"].value_counts()
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(cat_counts.index, cat_counts.values, color=sns.color_palette("pastel", len(cat_counts)))
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_xlabel("Count")
ax.set_title("Semantic Category Distribution")
ax.invert_yaxis()
plt.tight_layout()
save_and_show(fig, "semantic_category_distribution")

# Answer type
ans_counts = df["answer_type"].value_counts()
fig, ax = plt.subplots(figsize=(6, 5))
ax.pie(ans_counts.values, labels=ans_counts.index, autopct="%1.1f%%",
       colors=sns.color_palette("Set2", len(ans_counts)), startangle=140)
ax.set_title("Answer Type Distribution")
plt.tight_layout()
save_and_show(fig, "answer_type_distribution")

#Question & Answer length distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, label, color in zip(
    axes,
    ["q_len", "a_len"],
    ["Question length (words)", "Answer length (words)"],
    [FIG_COLS, "#DD8452"]
):
    ax.hist(df[col], bins=30, color=color, edgecolor="white", alpha=0.85)
    ax.axvline(df[col].median(), color="red", linestyle="--", linewidth=1.3,
               label=f"Median = {df[col].median():.0f}")
    ax.set_xlabel(label)
    ax.set_ylabel("Count")
    ax.set_title(f"Distribution of {label}")
    ax.legend()
plt.tight_layout()
save_and_show(fig, "04_length_distributions")

# Category × Answer-type heatmap
cross = pd.crosstab(df["category"], df["answer_type"])
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(cross, annot=True, fmt="d", cmap="YlOrRd", linewidths=0.4, ax=ax)
ax.set_title("Category × Answer Type Heatmap")
ax.set_xlabel("Answer Type")
ax.set_ylabel("Semantic Category")
plt.tight_layout()
save_and_show(fig, "05_category_answer_heatmap")
